# Exploration des données des barrages marocains

Notebook d'exploration des données nettoyées issues du pipeline ETL.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 100)

In [ ]:
# 1. Chargement des données
df = pd.read_parquet('../data/processed/barrages_clean.parquet')
print(f'Shape: {df.shape}')
print(f'Colonnes: {df.columns.tolist()}')
print(f'Barrages: {sorted(df["barrage"].unique())}')
print(f'Période: {df["date"].min()} → {df["date"].max()}')
df.head()

In [ ]:
# 2. Graphique évolution niveau par barrage
fig = px.line(df, x='date', y='niveau_pct', color='barrage',
              title='Évolution du taux de remplissage (%) par barrage',
              labels={'niveau_pct': 'Taux de remplissage (%)', 'date': 'Date', 'barrage': 'Barrage'},
              template='plotly_white')
fig.update_layout(height=500, hovermode='x unified')
fig.show()

In [ ]:
# 3. Statistiques par barrage
stats = []
for barrage in sorted(df['barrage'].unique()):
    sub = df[df['barrage'] == barrage].copy()
    sub = sub.sort_values('date')
    
    periode_debut = sub['date'].min()
    periode_fin = sub['date'].max()
    n_obs = len(sub)
    n_na = sub['niveau_pct'].isna().sum()
    pct_na = 100 * n_na / n_obs if n_obs > 0 else 0
    
    # Saisonnalité mensuelle (moyenne par mois)
    sub['month'] = sub['date'].dt.month
    monthly_avg = sub.groupby('month')['niveau_pct'].mean()
    seasonal_amplitude = monthly_avg.max() - monthly_avg.min() if len(monthly_avg) > 0 else np.nan
    
    stats.append({
        'barrage': barrage,
        'periode_debut': periode_debut,
        'periode_fin': periode_fin,
        'annees_historique': (periode_fin - periode_debut).days / 365.25,
        'n_obs': n_obs,
        'n_na': n_na,
        'pct_na': pct_na,
        'amplitude_saisonniere': seasonal_amplitude,
        'niveau_moyen': sub['niveau_pct'].mean(),
        'niveau_std': sub['niveau_pct'].std(),
    })

stats_df = pd.DataFrame(stats)
stats_df

In [ ]:
# 4. Tableau caractéristiques statiques
features = df[['barrage', 'capacite_max_m3', 'hauteur_m', 'annee_mise_en_service',
               'bassin_versant', 'latitude', 'longitude']].drop_duplicates()
features['capacite_Mm3'] = (features['capacite_max_m3'] / 1e6).round(1)
features = features[['barrage', 'bassin_versant', 'capacite_Mm3', 'hauteur_m',
                      'annee_mise_en_service', 'latitude', 'longitude']]
features

In [ ]:
# 5. Saisonnalité détaillée par barrage
fig = make_subplots(rows=2, cols=2, subplot_titles=sorted(df['barrage'].unique()))
for i, barrage in enumerate(sorted(df['barrage'].unique())):
    sub = df[df['barrage'] == barrage].copy()
    sub['month'] = sub['date'].dt.month
    monthly = sub.groupby('month')['niveau_pct'].mean().reset_index()
    row = i // 2 + 1
    col = i % 2 + 1
    fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['niveau_pct'],
                             mode='lines+markers', name=barrage, showlegend=False),
                  row=row, col=col)
fig.update_layout(height=600, title_text='Saisonnalité moyenne mensuelle par barrage')
fig.update_xaxes(title_text='Mois', tickmode='linear', tick0=1, dtick=1)
fig.update_yaxes(title_text='Niveau moyen (%)')
fig.show()

# VERDICT DONNÉES

## Combien d'années d'historique par barrage ?

D'après les statistiques ci-dessus, **tous les barrages ont seulement ~0.17 ans d'historique** (environ 2 mois : août-septembre 2026).

## L'historique est-il suffisant ?

❌ **NON — HISTORIQUE CRITIQUEMENT INSUFFISANT**

- Seuil minimum requis : **5 ans** (60 mois) pour capturer la saisonnalité et les cycles interannuels
- Historique disponible : **2 mois** (0.17 ans)
- **RISQUE N°1 DU PROJET CONFIRMÉ** : Impossible d'entraîner un modèle fiable avec si peu de données

## La saisonnalité est-elle visible ?

❌ **NON VISIBLE** — Avec seulement 2 mois de données (août-septembre), impossible de détecter un cycle annuel. L'amplitude saisonnière calculée est dérisoire car elle ne compare que 2 mois consécutifs.

## ACTIONS REQUISES

1. **Télécharger l'historique complet** depuis les bulletins mensuels PDF du Ministère (années 2000-2025)
2. **Ou utiliser ERA5 / données climatiques** comme proxy pour augmenter la profondeur temporelle
3. **Sans minimum 5 ans**, tout modèle sera du surapprentissage pur — **ARRÊT DU PROJET RECOMMANDÉ** jusqu'à résolution